In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Added to sys.path:", PROJECT_ROOT)

Added to sys.path: /Users/dimokritospsomatakis/Desktop/RES---Assignment-1


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.data_loader import load_rts24_from_csv
from src.models import Step1InputData, Step1MarketClearing, Step2InputData, Step2MarketClearing, Step3InputData, Step3MarketClearing, Step3ZonalInputData, Step3ZonalMarketClearing
from src.postprocess import step1_build_summary_tables, step2_build_summary_tables, step3_build_summary_tables, step3_zonal_build_summary_tables

Define data directory

In [7]:
DATA_DIR = Path("..") / "data"

data = load_rts24_from_csv(DATA_DIR)

print(f"Slack bus: {data.slack_bus}")
print(f"Number of generators: {len(data.generators)}")
print(f"Number of lines: {len(data.lines)}")

Slack bus: 13
Number of generators: 12
Number of lines: 34


# STEP 1

Choose hour for copper-plate market clearing

In [4]:
H = 19  # peak hour example

Extract hour-specific data

In [5]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))

# Wind availability
wind_farms_df = data.wind_farms.copy()
wind_av_h = data.wind_availability.query("hour == @H")

WINDS = wind_farms_df["wind_id"].tolist()
wind_avail = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
nodal_load_h = data.nodal_load.query("hour == @H")

LOAD_BUSES = nodal_load_h.query("demand_MW > 0")["bus"].tolist()
demand_pmax = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
bids_h = data.demand_bids.query("hour == @H")
demand_bid = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

Build input object

In [6]:
inp = Step1InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail=wind_avail,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
)

Solve Step 1 (Copper Plate Market Clearing)

In [7]:
model = Step1MarketClearing(inp, output_flag=0)
model.run()

Set parameter Username
Set parameter LicenseID to value 2785388
Academic license - for non-commercial use only - expires 2027-02-27


**Results**

In [8]:
df_gen, df_wind, df_dem, totals = step1_build_summary_tables(model.results, inp)

print("Market price:", round(totals["market_price"], 2), "USD/MWh")
print("Objective (Social Welfare):", round(totals["objective_welfare"], 2))
print("Total operating cost:", round(totals["total_operating_cost"], 2))

print("\nTotal generation:", round(totals["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals["total_demand_served_MW"], 2), "MW")

Market price: 13.32 USD/MWh
Objective (Social Welfare): 619868.95
Total operating cost: 16251.05

Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW


Generator results

In [9]:
df_gen

,generator,p_MW,marginal_cost,profit
0,1,99.185514,13.32,0.0
1,2,0.000000,13.32,0.0
2,3,0.000000,20.70,-0.0
3,4,0.000000,20.93,-0.0
4,5,0.000000,26.11,-0.0
5,6,155.000000,10.52,434.0
6,7,155.000000,10.52,434.0
7,8,400.000000,6.02,2920.0
8,9,400.000000,5.47,3140.0
9,10,300.000000,0.00,3996.0


Wind results

In [10]:
df_wind

,wind,p_MW,profit
0,1,73.622733,980.654803
1,2,83.964245,1118.403746
2,3,80.763464,1075.769341
3,4,82.379405,1097.293675
4,5,73.687627,981.519194
5,6,86.897011,1157.468190


Demand results

In [11]:
df_dem

,bus,d_MW,bid_price,utility
0,1,100.7190,240.0,22830.98292
1,2,90.1170,240.0,20427.72156
2,3,166.9815,240.0,37851.36642
3,4,68.9130,240.0,15621.19884
4,5,66.2625,240.0,15020.38350
5,6,127.2240,240.0,28839.13632
6,7,116.6220,240.0,26435.87496
7,8,159.0300,240.0,36048.92040
8,9,161.6805,240.0,36649.73574
9,10,180.2340,240.0,40855.44312


Welfare breakdown check

In [12]:
print("Total generator profit:", round(totals["total_gen_profit"], 2))
print("Total wind profit:", round(totals["total_wind_profit"], 2))
print("Total demand utility:", round(totals["total_demand_utility"], 2))

print("\nCheck: Utility - Cost = Welfare")
print(round(
    totals["total_demand_utility"]
    + totals["total_gen_profit"]
    + totals["total_wind_profit"],
    6
))

Total generator profit: 12642.5
Total wind profit: 6411.11
Total demand utility: 600815.34

Check: Utility - Cost = Welfare
619868.948948


# STEP 2

In [7]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))

# Hours (all 24)
hours = [int(h) for h in sorted(data.wind_availability["hour"].unique())]

# Wind availability
wind_farms_df = data.wind_farms.copy()
WINDS = wind_farms_df["wind_id"].tolist()

wind_avail_t = {}

for h in hours:
    wind_av_h = data.wind_availability.query("hour == @h")
    wind_avail_t[h] = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
LOAD_BUSES = data.nodal_load.query("demand_MW > 0")["bus"].unique().tolist()

demand_pmax_t = {}
for h in hours:
    nodal_load_h = data.nodal_load.query("hour == @h")
    demand_pmax_t[h] = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
demand_bid_t = {}
for h in hours:
    bids_h = data.demand_bids.query("hour == @h")
    demand_bid_t[h] = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

In [8]:
inpu = Step2InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail_t=wind_avail_t,
    demand_pmax_t=demand_pmax_t,
    demand_bid_t=demand_bid_t,
    hours=hours                   
)
print("hours:", hours, type(hours))


hours: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24] <class 'list'>


In [9]:
model2 = Step2MarketClearing(inpu, output_flag=0)
model2.run()

Set parameter Username
Set parameter LicenseID to value 2775062
Academic license - for non-commercial use only - expires 2027-02-04


In [10]:
df_gen, df_wind, df_dem, df_storage, df_prices, totals = step2_build_summary_tables(model2.results, inpu)


In [11]:
print("STEP 2 - COPPER PLATE 24H WITH STORAGE")

print("\n1. HOURLY PRICES")
print(df_prices.round(2))

print("\n2. TOTALS 24 HOURS")
print(pd.Series(totals).round(2))

print("\n3. GENERATOR TOTALS & PROFITS")
print(df_gen.round(2))

print("\n4. WIND TOTALS & PROFITS") 
print(df_wind.round(2))

print("\n5. STORAGE OPERATION (Hourly)")
print(df_storage.round(2))

print("\n6. DEMAND TOTALS & UTILITY")
print(df_dem.round(2))


STEP 2 - COPPER PLATE 24H WITH STORAGE

1. HOURLY PRICES
    hour  price
0      1  10.52
1      2  10.52
2      3  10.52
3      4  10.52
4      5  10.52
5      6  10.52
6      7  10.52
7      8  10.89
8      9  10.89
9     10  10.89
10    11  10.89
11    12  10.89
12    13  10.89
13    14  10.89
14    15  10.89
15    16  10.89
16    17  12.57
17    18  12.57
18    19  12.57
19    20  10.89
20    21  10.89
21    22  10.52
22    23  10.52
23    24  10.52

2. TOTALS 24 HOURS
objective_welfare_24h       11211650.47
total_operating_cost_24h      257593.13
total_gen_MWh                  40311.46
total_wind_MWh                 12506.05
total_demand_served_MWh        52771.46
total_gen_profit_24h          186758.31
total_wind_profit_24h         137151.04
total_demand_utility_24h    10887741.11
total_storage_profit_24h           0.00
avg_price                         10.95
max_price                         12.57
min_price                         10.52
dtype: float64

3. GENERATOR TOTALS & PROF

# STEP 3

In [8]:
# STEP 3A - Build nodal input data


H = 19   # same representative hour as Step 1

# Generators
gen_df = data.generators.copy()
GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))
generator_bus = dict(zip(gen_df["unit"], gen_df["bus"]))

# Wind
wind_farms_df = data.wind_farms.copy()
wind_av_h = data.wind_availability.query("hour == @H")

WINDS = wind_farms_df["wind_id"].tolist()
wind_avail = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))
wind_bus = dict(zip(wind_farms_df["wind_id"], wind_farms_df["bus"]))

# Demand
nodal_load_h = data.nodal_load.query("hour == @H")
LOAD_BUSES = nodal_load_h.query("demand_MW > 0")["bus"].tolist()
demand_pmax = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))

bids_h = data.demand_bids.query("hour == @H")
demand_bid = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

# Buses
BUSES = data.buses

# Lines
lines_df = data.lines.copy().reset_index(drop=True)
LINES = lines_df.index.tolist()

line_from = dict(zip(LINES, lines_df["from_bus"]))
line_to = dict(zip(LINES, lines_df["to_bus"]))
line_B = {l: 1 / lines_df.loc[l, "reactance_pu"] for l in LINES}
line_fmax = dict(zip(LINES, lines_df["capacity_modified_MW"]))

inp3 = Step3InputData(
    BUSES=BUSES,
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    LINES=LINES,
    slack_bus=data.slack_bus,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    generator_bus=generator_bus,
    wind_avail=wind_avail,
    wind_bus=wind_bus,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
    line_from=line_from,
    line_to=line_to,
    line_B=line_B,
    line_fmax=line_fmax,
)

print("Buses:", len(BUSES))
print("Lines:", len(LINES))
print("Slack bus:", data.slack_bus)
print("Example susceptance:", round(line_B[0], 3))
print("Example capacity:", line_fmax[0])

Buses: 24
Lines: 34
Slack bus: 13
Example susceptance: 68.493
Example capacity: 175.0


In [9]:
# STEP 3A - Solve nodal model


model3 = Step3MarketClearing(inp3, output_flag=0)
model3.run()

df_gen3, df_wind3, df_dem3, df_bus3, df_line3, totals3 = step3_build_summary_tables(model3.results, inp3)

print("STEP 3A - NODAL MARKET CLEARING")
print("Objective (Social Welfare):", round(totals3["objective_welfare"], 2))
print("Total operating cost:", round(totals3["total_operating_cost"], 2))
print("Total generation:", round(totals3["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals3["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals3["total_demand_served_MW"], 2), "MW")
print("Min nodal price:", round(totals3["min_nodal_price"], 2), "USD/MWh")
print("Max nodal price:", round(totals3["max_nodal_price"], 2), "USD/MWh")
print("Price spread:", round(totals3["price_spread"], 2), "USD/MWh")
print("Congested lines:", totals3["n_congested_lines"])

Set parameter Username
Set parameter LicenseID to value 2775062
Academic license - for non-commercial use only - expires 2027-02-04
STEP 3A - NODAL MARKET CLEARING
Objective (Social Welfare): 615455.7
Total operating cost: 20664.3
Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW
Min nodal price: 5.47 USD/MWh
Max nodal price: 29.45 USD/MWh
Price spread: 23.98 USD/MWh
Congested lines: 3


In [12]:
# display nodal results
df_bus3.round(3)

,bus,theta_rad,nodal_price,gen_MW,wind_MW,demand_MW,net_injection_MW
0,1,-2.254,19.709,152.000,0.000,100.719,51.281
1,2,-2.713,19.809,152.000,0.000,90.117,61.883
2,3,-1.803,16.545,0.000,73.623,166.982,-93.359
3,4,-8.625,20.109,0.000,0.000,68.913,-68.913
4,5,-4.231,20.365,0.000,83.964,66.262,17.702
5,6,-12.916,20.750,0.000,0.000,127.224,-127.224
6,7,14.233,20.700,264.972,80.763,116.622,229.113
7,8,-0.705,20.700,0.000,0.000,159.030,-159.030
8,9,-5.815,20.355,0.000,0.000,161.680,-161.680
9,10,-7.944,21.045,0.000,0.000,180.234,-180.234


In [13]:
df_line3.round(3)

,line,from_bus,to_bus,B,flow_MW,Fmax_MW,loading_pct,congested
0,0,1,2,68.493,31.484,175.0,17.991,False
1,1,11,13,20.492,-43.492,500.0,8.698,False
2,2,1,3,4.439,-2.000,175.0,1.143,False
3,3,11,14,23.474,-69.766,500.0,13.953,False
4,4,1,5,11.025,21.797,350.0,6.228,False
5,5,12,13,20.492,22.795,500.0,4.559,False
6,6,2,4,7.375,43.597,175.0,24.913,False
7,7,12,23,10.152,-213.072,500.0,42.614,False
8,8,2,6,4.878,49.770,175.0,28.440,False
9,9,13,23,11.312,-250.000,250.0,100.000,True


In [14]:
df_gen3.round(3)

,generator,bus,p_MW,marginal_cost,nodal_price,profit
0,1,1,152.000,13.32,19.709,971.155
1,2,2,152.000,13.32,19.809,986.265
2,3,7,264.972,20.70,20.700,0.000
3,4,13,17.194,20.93,20.930,0.000
4,5,15,0.000,26.11,10.520,-0.000
5,6,15,13.786,10.52,10.520,0.000
6,7,16,0.000,10.52,8.666,-0.000
7,8,18,400.000,6.02,6.198,71.374
8,9,21,209.234,5.47,5.470,0.000
9,10,22,300.000,0.00,6.066,1819.903


In [15]:
df_wind3.round(3)

,wind,bus,p_MW,nodal_price,profit
0,1,3,73.623,16.545,1218.120
1,2,5,83.964,20.365,1709.955
2,3,7,80.763,20.700,1671.804
3,4,16,82.379,8.666,713.865
4,5,21,73.688,5.470,403.071
5,6,23,86.897,13.528,1175.547


In [ ]:
df_dem3.round(3)

In [16]:
# nodal sensitivity analysis

from copy import deepcopy

def run_step3_line_sensitivity(inp_base, tested_line, tested_capacities):
    rows = []

    for cap in tested_capacities:
        inp_tmp = deepcopy(inp_base)
        inp_tmp.line_fmax[tested_line] = cap

        model_tmp = Step3MarketClearing(inp_tmp, output_flag=0)
        model_tmp.run()

        _, _, _, _, _, totals_tmp = step3_build_summary_tables(model_tmp.results, inp_tmp)

        rows.append({
            "line": tested_line,
            "tested_capacity_MW": cap,
            "min_nodal_price": totals_tmp["min_nodal_price"],
            "max_nodal_price": totals_tmp["max_nodal_price"],
            "price_spread": totals_tmp["price_spread"],
            "social_welfare": totals_tmp["objective_welfare"],
            "n_congested_lines": totals_tmp["n_congested_lines"],
        })

    return pd.DataFrame(rows)

#To identify the line index for one of your tightened lines
lines_df[["from_bus", "to_bus", "capacity_modified_MW"]]

# Then run, for example, on one of the modified lines 

sens_df = run_step3_line_sensitivity(
    inp_base=inp3,
    tested_line=11,  #line 14-16
    tested_capacities=[150, 200, 250, 300, 400]
)

sens_df.round(3)

,line,tested_capacity_MW,min_nodal_price,max_nodal_price,price_spread,social_welfare,n_congested_lines
0,11,150,4.831,35.751,30.920,611467.555,1
1,11,200,5.548,35.090,29.541,613611.796,1
2,11,250,5.470,29.447,23.977,615455.699,3
3,11,300,5.470,25.032,19.562,616882.518,3
4,11,400,5.470,20.930,15.460,617547.900,2


In [17]:
# STEP 3B - Build zonal input data
ZONES = ["Z1", "Z2"]
BUS_ZONE = {b: ("Z1" if b <= 15 else "Z2") for b in BUSES}

# One interface between the two zones
INTERFACES = [("Z1", "Z2")]

# ATC = total capacity of all crossing lines (allowed by assignment)
crossing_lines = [
    l for l in LINES
    if BUS_ZONE[line_from[l]] != BUS_ZONE[line_to[l]]
]

atc = {
    ("Z1", "Z2"): sum(line_fmax[l] for l in crossing_lines)
}

print("Crossing lines:", crossing_lines)
print("ATC Z1-Z2:", atc[("Z1", "Z2")], "MW")

inp3z = Step3ZonalInputData(
    ZONES=ZONES,
    BUS_ZONE=BUS_ZONE,
    INTERFACES=INTERFACES,
    atc=atc,
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    generator_bus=generator_bus,
    wind_avail=wind_avail,
    wind_bus=wind_bus,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
)

Crossing lines: [7, 9, 11, 12, 13, 15, 17]
ATC Z1-Z2: 2800.0 MW


In [21]:
# STEP 3B - Solve zonal model


model3z = Step3ZonalMarketClearing(inp3z, output_flag=0)
model3z.run()

df_gen3z, df_wind3z, df_dem3z, df_zone3z, df_transfer3z, totals3z = step3_zonal_build_summary_tables(model3z.results, inp3z)

print("STEP 3B - ZONAL MARKET CLEARING")
print("Objective (Social Welfare):", round(totals3z["objective_welfare"], 2))
print("Total operating cost:", round(totals3z["total_operating_cost"], 2))
print("Total generation:", round(totals3z["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals3z["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals3z["total_demand_served_MW"], 2), "MW")
print("Zonal prices:", {z: round(model3z.results.zonal_prices[z], 2) for z in ZONES})

df_zone3z.round(3)
df_transfer3z.round(3)
df_gen3z.round(3)
df_wind3z.round(3)
df_dem3z.round(3)

STEP 3B - ZONAL MARKET CLEARING
Objective (Social Welfare): 619868.95
Total operating cost: 16251.05
Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW
Zonal prices: {'Z1': 13.32, 'Z2': 13.32}


,bus,zone,d_MW,bid_price,zonal_price,utility
0,1,Z1,100.719,240.0,13.32,22830.983
1,2,Z1,90.117,240.0,13.32,20427.722
2,3,Z1,166.982,240.0,13.32,37851.366
3,4,Z1,68.913,240.0,13.32,15621.199
4,5,Z1,66.262,240.0,13.32,15020.384
5,6,Z1,127.224,240.0,13.32,28839.136
6,7,Z1,116.622,240.0,13.32,26435.875
7,8,Z1,159.030,240.0,13.32,36048.920
8,9,Z1,161.680,240.0,13.32,36649.736
9,10,Z1,180.234,240.0,13.32,40855.443


In [22]:
compare_df = pd.DataFrame([
    {
        "framework": "Nodal",
        "social_welfare": totals3["objective_welfare"],
        "total_gen_profit": totals3["total_gen_profit"],
        "total_wind_profit": totals3["total_wind_profit"],
        "total_demand_utility": totals3["total_demand_utility"],
        "price_info": f'{round(totals3["min_nodal_price"],2)} to {round(totals3["max_nodal_price"],2)}'
    },
    {
        "framework": "Zonal",
        "social_welfare": totals3z["objective_welfare"],
        "total_gen_profit": totals3z["total_gen_profit"],
        "total_wind_profit": totals3z["total_wind_profit"],
        "total_demand_utility": totals3z["total_demand_utility"],
        "price_info": str({z: round(model3z.results.zonal_prices[z], 2) for z in ZONES})
    }
])

compare_df

,framework,social_welfare,total_gen_profit,total_wind_profit,total_demand_utility,price_info
0,Nodal,615455.699274,5704.506554,6892.361286,591817.448586,5.47 to 29.45
1,Zonal,619868.948948,12642.500000,6411.108948,600815.340000,"{'Z1': 13.32, 'Z2': 13.32}"


In [23]:
def dc_flow_from_injections(inp_nodal, injections):
    """
    Solve DC load flow from fixed net injections using the original nodal network.
    """
    buses = inp_nodal.BUSES
    idx = {b: i for i, b in enumerate(buses)}
    n = len(buses)

    Bbus = np.zeros((n, n))

    for l in inp_nodal.LINES:
        i = idx[inp_nodal.line_from[l]]
        j = idx[inp_nodal.line_to[l]]
        b = inp_nodal.line_B[l]

        Bbus[i, i] += b
        Bbus[j, j] += b
        Bbus[i, j] -= b
        Bbus[j, i] -= b

    slack_i = idx[inp_nodal.slack_bus]
    keep = [i for i in range(n) if i != slack_i]

    P = np.array([injections.get(b, 0.0) for b in buses], dtype=float)

    theta = np.zeros(n)
    theta_red = np.linalg.solve(Bbus[np.ix_(keep, keep)], P[keep])
    theta[keep] = theta_red

    flows = {}
    for l in inp_nodal.LINES:
        i = idx[inp_nodal.line_from[l]]
        j = idx[inp_nodal.line_to[l]]
        flows[l] = inp_nodal.line_B[l] * (theta[i] - theta[j])

    theta_dict = {b: theta[idx[b]] for b in buses}
    return theta_dict, flows

zonal_injections = {}
for b in BUSES:
    gen_b = sum(model3z.results.pG[g] for g in GENERATORS if generator_bus[g] == b)
    wind_b = sum(model3z.results.pW[w] for w in WINDS if wind_bus[w] == b)
    dem_b = model3z.results.d[b] if b in LOAD_BUSES else 0.0
    zonal_injections[b] = gen_b + wind_b - dem_b

theta_zonal_dc, flows_zonal_dc = dc_flow_from_injections(inp3, zonal_injections)

feas_rows = []
for l in LINES:
    overload = max(0.0, abs(flows_zonal_dc[l]) - line_fmax[l])
    same_zone = BUS_ZONE[line_from[l]] == BUS_ZONE[line_to[l]]

    feas_rows.append({
        "line": l,
        "from_bus": line_from[l],
        "to_bus": line_to[l],
        "same_zone": same_zone,
        "flow_from_zonal_dispatch_MW": flows_zonal_dc[l],
        "Fmax_MW": line_fmax[l],
        "overload_MW": overload
    })

df_feas = pd.DataFrame(feas_rows)
df_internal_overload = df_feas[(df_feas["same_zone"]) & (df_feas["overload_MW"] > 1e-5)].copy()

print("Total overloaded internal lines:", len(df_internal_overload))
print("Max internal overload (MW):", round(df_internal_overload["overload_MW"].max(), 4) if len(df_internal_overload) > 0 else 0.0)
print("Sum of internal overloads (MW):", round(df_internal_overload["overload_MW"].sum(), 4))

df_internal_overload.round(3)

Total overloaded internal lines: 0
Max internal overload (MW): 0.0
Sum of internal overloads (MW): 0.0


,line,from_bus,to_bus,same_zone,flow_from_zonal_dispatch_MW,Fmax_MW,overload_MW


In [24]:
df_all_overload = df_feas[df_feas["overload_MW"] > 1e-5].copy()

print("Total overloaded lines:", len(df_all_overload))
print("Max overload (MW):", round(df_all_overload["overload_MW"].max(), 4) if len(df_all_overload) > 0 else 0.0)
print("Sum of overloads (MW):", round(df_all_overload["overload_MW"].sum(), 4))

df_all_overload.round(3)

Total overloaded lines: 3
Max overload (MW): 172.6309
Sum of overloads (MW): 356.2427


,line,from_bus,to_bus,same_zone,flow_from_zonal_dispatch_MW,Fmax_MW,overload_MW
9,9,13,23,False,-330.841,250.0,80.841
11,11,14,16,False,-422.631,250.0,172.631
15,15,15,21,False,-502.770,400.0,102.770
